<a href="https://colab.research.google.com/github/abaziri/signate-Jleague/blob/main/tutorials/SIGNATE_Tutorial_for_Colaboratory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SIGNATE Tutorial Notebook for Colaboratory

Google Colaboratoryを利用してSIGNATE ○○コンペティションに参加する際のチュートリアルノートブックです。


# 事前準備： SIGNATE CLI

[SIGNATE CLIとは](https://pypi.org/project/signate/)

## Installation

SIGNATEコンペティションのデータセットのダウンロードやサブミットする際に便利なCLIツールをインストールします。

In [20]:
! pip install signate

## Configuration

SIGNATE CLIを使うためには、最初にTokenファイルが必要です。

[SIGNATEアカウント設定ページ](https://signate.jp/account_settings) のおAPI Tokenファイル（signate.json）を発行し、ローカルPCにダウンロードします。

### Upload token file

ローカルPCにダウンロードしたTokenファイルをGoogle Colaboratoryにアップロードします。

In [21]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

KeyboardInterrupt: 

In [ ]:
!signate token --email=nagovin@icloud.com

### Move token file

TokenファイルをSIGNATE CLIが認識できるように `$HOME/.signate/signate.json` に移動します。

In [ ]:
! mkdir -p /root/.signate && mv signate.json /root/.signate/

## Check command

ヘルプコマンド実行

In [ ]:
! signate --help

# コンペ参加

## 開催中のコンペ一覧

In [ ]:
! signate list

## データセットダウンロード

### ファイル一覧取得

In [ ]:
! signate files -c 102

### 一括ファイルダウンロード

`competitonId: 102` の一括ダウンロード

In [ ]:
! signate download -c 102

In [ ]:
! ls -al

### 個別ファイルダウンロード

`competitionId: 102, fileId: 272` train.tsv ダウンロードする場合

In [ ]:
! signate download -c 102 -f 272

## データ分析

In [ ]:
# python バージョン確認
! python -V

In [ ]:
# インストール済みライブラリー確認
! pip freeze

In [ ]:
# ここから分析コードを書く
import pandas as pd
train = pd.read_csv('/train.csv')
train_add = pd.read_csv('/train_add.csv')
condition = pd.read_csv('/condition.csv')
condition_add = pd.read_csv('/condition_add.csv')
stadium = pd.read_csv('/stadium.csv')
year_2014_add = pd.read_csv('/2014_add.csv')
test = pd.read_csv('/test.csv')


In [ ]:
# check the detail of train and train add data
display(train.head(3), train_add)

In [ ]:
# check the detail of condition and condition add data
display(condition.head(3), condition_add.head(3))

In [ ]:
# check the detail of stadium and 2014 data
display(stadium.head(3), year_2014_add.head(3))

In [ ]:
# concat with train and train add data, and condition and conditon add data
full_train = pd.concat([train, train_add])
full_condition = pd.concat([condition, condition_add])




In [ ]:
# 結合前と後で形に異常がないか確認
print('train concat')
print('before: ', train.shape, train_add.shape)
print('after: ', full_train.shape)

print('condition concat')
print('before: ', condition.shape, condition_add.shape)
print('after: ', full_condition.shape)



In [ ]:
# stadium を再ロードして確実に初期状態から処理する
stadium = pd.read_csv('/stadium.csv')

# rename the name column to stadium
stadium = stadium.rename(columns={'name': 'stadium'})

# extract the only stadium and capa columns to merge into a data
stadium = stadium[['stadium', 'capa']]

# 結合済みのデータフレーム full_condition から必要な列のみを抽出する（スペルミスを修正）
full_condition = full_condition[['id', 'home_score', 'away_score', 'weather', 'temperature', 'humidity', 'referee', 'home_team']]

signate チュートリアルではfull_conditionの抽出コラムは「full_condition = full_condition[['id' ,'weather' ,'temperature' ,'humidity']]」であるが自分は特徴量を増やして行う。
＊追加コラム
'home_score', 'away_score','referee', 'home_team'


In [30]:
# 1. stadiumの列名が 'name' の場合、'stadium' に確実に変換する
if 'name' in stadium.columns:
    stadium = stadium.rename(columns={'name': 'stadium'})

# 2. 訓練データ (full_train) に stadium と full_condition を結合
# 何度実行しても大丈夫なように列の存在チェックを追加
if 'capa' not in full_train.columns:
    full_train = pd.merge(full_train, stadium, on='stadium', how='left')

if 'weather' not in full_train.columns:
    full_train = pd.merge(full_train, full_condition, on='id', how='left')

# 3. テストデータに対しても同様に結合を行う
full_test = test.copy()
if 'capa' not in full_test.columns:
    full_test = pd.merge(full_test, stadium, on='stadium', how='left')

if 'weather' not in full_test.columns:
    full_test = pd.merge(full_test, full_condition, on='id', how='left')

# if the test data contains 'address'columns, drop that columns since it is not contained in train data
if 'address' in full_test.columns:
    full_test = full_test.drop(columns = ['address'])

# 結合後の結果を少し確認する
display(full_train.head(3))
display(full_test.head(3))

,id,y,year,stage,match,gameday,time,home,away,stadium,tv,capa,home_score,away_score,weather,temperature,humidity,referee,home_team
0,13994,18250,2012,Ｊ１,第１節第１日,03/10(土),14:04,ベガルタ仙台,鹿島アントラーズ,ユアテックスタジアム仙台,スカパー／ｅ２／スカパー光／ＮＨＫ総合,19694,1,0,雨,3.8,66%,木村 博之,ベガルタ仙台
1,13995,24316,2012,Ｊ１,第１節第１日,03/10(土),14:04,名古屋グランパス,清水エスパルス,豊田スタジアム,スカパー／ｅ２／スカパー光（Ｊ ＳＰＯＲＴＳ ４）／ＮＨＫ名古屋,40000,1,0,屋内,12.4,43%,西村 雄一,名古屋グランパス
2,13996,17066,2012,Ｊ１,第１節第１日,03/10(土),14:04,ガンバ大阪,ヴィッセル神戸,万博記念競技場,スカパー／ｅ２／スカパー光（Ｊ ＳＰＯＲＴＳ １）／ＮＨＫ大阪,21000,2,3,晴一時雨,11.3,41%,高山 啓義,ガンバ大阪


,id,year,stage,match,gameday,time,home,away,stadium,tv,capa,home_score,away_score,weather,temperature,humidity,referee,home_team
0,15822,2014,Ｊ１,第１８節第１日,08/02(土),19:04,ベガルタ仙台,大宮アルディージャ,ユアテックスタジアム仙台,スカパー！／スカパー！プレミアムサービス,19694,2,2,晴,27.4,70%,中村 太,ベガルタ仙台
1,15823,2014,Ｊ１,第１８節第１日,08/02(土),18:34,鹿島アントラーズ,サンフレッチェ広島,県立カシマサッカースタジアム,スカパー！／スカパー！プレミアムサービス,40728,5,1,晴,30.8,65%,扇谷 健司,鹿島アントラーズ
2,15824,2014,Ｊ１,第１８節第１日,08/02(土),19:04,浦和レッズ,ヴィッセル神戸,埼玉スタジアム２００２,スカパー！／スカパー！プレミアムサービス／ＮＨＫ ＢＳ１／テレ玉,63700,2,2,晴,31.7,58%,松尾 一,浦和レッズ


In [31]:
print('full train shape:', full_train.shape)
print('full test shape:', full_test.shape)

full train shape: (1953, 19)
full test shape: (313, 18)


## 分析結果サブミット

- サブミットの結果は評価が終わるとメールにて通知される
- `competitionId: 102` にメモを添えてサブミットする場合

In [ ]:
! signate submit -c 102 -n "submit from Google Colaboratory" sample_submit.tsv

# その他のリソース

- [SIGNATE コンペ一覧](https://signate.jp/competitions)
- [SIGNATE CLIの概要](https://pypi.org/project/signate)
- [SIGNATEアカウント設定ページ](https://signate.jp/account_settings)
- [Colaboratory の概要](/notebooks/basic_features_overview.ipynb)
- [markdown のガイド](/notebooks/markdown_guide.ipynb)
- [ライブラリのインポートと依存関係のインストール](/notebooks/snippets/importing_libraries.ipynb)
 [Colab の TensorFlow 2](/notebooks/tensorflow_version.ipynb)